# SqueezeNet 1.1 no HEp-2: comparação de Adam e SGD

Este notebook executa os seis cenários com 30 épocas e batch size 32. O treino usa `weight_decay=2e-4`, LR inicial `1e-3` e uma validação estratificada de 10% do conjunto de treino para guardar o melhor checkpoint. No SGD, a taxa decai linearmente.

O artigo original da SqueezeNet usa `weight_decay=2e-4`; seus hiperparâmetros de ImageNet não são diretamente transferidos para HEp-2. Use a seção de validação de LR antes da comparação final. Carregue a versão atualizada do repositório em `/content/Projeto-PDI` e ative a GPU.


In [ ]:
!nvidia-smi

In [ ]:
%cd /content/Projeto-PDI
!pip install -q scikit-learn pandas pillow
import torch, torchvision
print("PyTorch:", torch.__version__, "TorchVision:", torchvision.__version__)
assert torch.cuda.is_available(), "Ative uma GPU em Ambiente de execução > Alterar tipo de ambiente de execução."

## Comparação de learning rate (SGD + equalização)

Este é o cenário SGD com pior resultado na linha de base. Execute estas três células com a mesma seed e compare a melhor `Val Acc`. `--skip-test` evita avaliar o conjunto de teste durante a escolha. Depois, use a taxa vencedora nos comandos SGD da comparação final.


In [ ]:
!python -u main_squeezenet.py --preprocessing histogram_equalization --optimizer sgd --lr 0.001 --weight-decay 0.0002 --seed 42 --epochs 30 --batch-size 32 --amp --num-workers 2 --skip-test

In [ ]:
!python -u main_squeezenet.py --preprocessing histogram_equalization --optimizer sgd --lr 0.003 --weight-decay 0.0002 --seed 42 --epochs 30 --batch-size 32 --amp --num-workers 2 --skip-test

In [ ]:
!python -u main_squeezenet.py --preprocessing histogram_equalization --optimizer sgd --lr 0.01 --weight-decay 0.0002 --seed 42 --epochs 30 --batch-size 32 --amp --num-workers 2 --skip-test

Depois de comparar as três saídas, defina `SGD_LR` com a taxa de maior `Val Acc`. Os comandos SGD abaixo usam essa variável; execute as seis células finais depois de escolher a taxa.


In [ ]:
SGD_LR = 0.001  # substitua pela taxa com melhor Val Acc

## Imagens originais

In [ ]:
!python -u main_squeezenet.py --preprocessing none --optimizer adam --lr 0.001 --weight-decay 0.0002 --seed 42 --epochs 30 --batch-size 32 --amp --num-workers 2

In [ ]:
!python -u main_squeezenet.py --preprocessing none --optimizer sgd --lr {SGD_LR} --weight-decay 0.0002 --seed 42 --epochs 30 --batch-size 32 --amp --num-workers 2

## Expansão de contraste

In [ ]:
!python -u main_squeezenet.py --preprocessing contrast_expansion --optimizer adam --lr 0.001 --weight-decay 0.0002 --seed 42 --epochs 30 --batch-size 32 --amp --num-workers 2

In [ ]:
!python -u main_squeezenet.py --preprocessing contrast_expansion --optimizer sgd --lr {SGD_LR} --weight-decay 0.0002 --seed 42 --epochs 30 --batch-size 32 --amp --num-workers 2

## Equalização de histograma

In [ ]:
!python -u main_squeezenet.py --preprocessing histogram_equalization --optimizer adam --lr 0.001 --weight-decay 0.0002 --seed 42 --epochs 30 --batch-size 32 --amp --num-workers 2

In [ ]:
!python -u main_squeezenet.py --preprocessing histogram_equalization --optimizer sgd --lr {SGD_LR} --weight-decay 0.0002 --seed 42 --epochs 30 --batch-size 32 --amp --num-workers 2